*Trabajo Práctico Nro1 - Primer Encuentro con la EPH*

- Grupo 5 del curso "M73 13 Seminarios Optativos - Taller de Programación"
- Integrantes: Santiago Caballero Yver, Oriana Valentina Gegena y Valeria Eliana Gonzalez.


In [4]:
!pip install openpyxl pandas
# Cargamos librerias a utilizar
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker


# Cargamos bases de datos T424 y T425:
T424 = pd.read_excel('C:/Users/gegen/OneDrive/Documentos/2. Economía Aplicada/Programación/usu_individual_T424.xlsx')
T425 = pd.read_excel('C:/Users/gegen/OneDrive/Documentos/2. Economía Aplicada/Programación/usu_individual_T425.xlsx')

In [6]:
# Variables de interés a utilizar en el analisis
variables = [
    "CODUSU", "ANO4", "TRIMESTRE", "NRO_HOGAR", "PONDERA", 
    "CH04", "CH06", "CH07", "CH08", "NIVEL_ED", 
    "ESTADO", "CAT_OCUP", "EMPLEO", "SECTOR", 
    "PP04C", "PP04D_COD", "P21", "P47T", 
    "PP07H", "PP3E_TOT"
]

T424_v = T424[variables].copy()
T425_v = T425[variables].copy()
    
# Variables incluidas:
# "PP07H"     Descuento jubilatorio (formalidad asalariada)
# "PP3E_TOT"  Horas trabajadas (salario horario)

#Seguimiento de variables:
#CH04 - Varón/Mujer
#CH06 - edad
#CH07 - Estado civil
#CH08 - Cobertura salud
#PP04C - Cantidad de personas que trabajan en el establecimiento
#PP04D_COD - Codigo de ocupación
#P21 - ingreso de la ocupación principal
#P47T - Ingreso total individual



In [7]:
# Resumen descriptivo de las bases para observar posibles outliers
resumen_2024 = T424_v.describe().T
resumen_2025 = T425_v.describe().T

# Mostramos el resultado del 2024
print("Resumen estadístico EPH - Cuarto Trimestre 2024:")
display(resumen_2024) 

# Mostramos el resultado del 2025
print("\nResumen estadístico EPH - Cuarto Trimestre 2025:")
display(resumen_2025)

Resumen estadístico EPH - Cuarto Trimestre 2024:


,count,mean,std,min,25%,50%,75%,max
ANO4,46860.0,2024.000000,0.000000,2024.0,2024.0,2024.0,2024.0,2024.0
TRIMESTRE,46860.0,4.000000,0.000000,4.0,4.0,4.0,4.0,4.0
NRO_HOGAR,46860.0,1.034550,0.634871,1.0,1.0,1.0,1.0,72.0
PONDERA,46860.0,635.586278,824.688160,23.0,180.0,315.0,663.0,7989.0
CH04,46860.0,1.519718,0.499616,1.0,1.0,2.0,2.0,2.0
CH06,46860.0,36.759112,22.182034,-1.0,18.0,35.0,54.0,103.0
CH07,46860.0,3.521618,1.651600,1.0,2.0,5.0,5.0,9.0
CH08,46860.0,2.134998,1.869091,1.0,1.0,1.0,4.0,123.0
NIVEL_ED,46860.0,3.780644,1.756795,1.0,3.0,4.0,5.0,7.0
ESTADO,46860.0,2.181946,1.133963,0.0,1.0,3.0,3.0,4.0



Resumen estadístico EPH - Cuarto Trimestre 2025:


,count,mean,std,min,25%,50%,75%,max
ANO4,43703.0,2025.000000,0.000000,2025.0,2025.0,2025.0,2025.0,2025.0
TRIMESTRE,43703.0,4.000000,0.000000,4.0,4.0,4.0,4.0,4.0
NRO_HOGAR,43703.0,1.039311,0.840685,1.0,1.0,1.0,1.0,71.0
PONDERA,43703.0,687.200993,905.356807,24.0,192.0,342.0,724.0,7954.0
CH04,43703.0,1.525639,0.499348,1.0,1.0,2.0,2.0,2.0
CH06,43703.0,37.586573,22.326015,-1.0,19.0,36.0,55.0,101.0
CH07,43703.0,3.513946,1.649546,1.0,2.0,5.0,5.0,9.0
CH08,43703.0,2.159714,1.768585,1.0,1.0,1.0,4.0,23.0
NIVEL_ED,43703.0,3.751504,1.737705,1.0,3.0,4.0,5.0,7.0
ESTADO,43703.0,2.168021,1.121546,0.0,1.0,3.0,3.0,4.0


In [10]:
# Reemplazamos los datos correspondientes a NS/NC a NaN
def reemplazar_nsnc_por_nan(df):
    df_clean = df.copy()

    # Variables categóricas donde 9 es NS/NC
    vars_cat = ['CAT_OCUP', 'EMPLEO', 'SECTOR', 'CH07']
    for col in vars_cat:
        df_clean[col] = df_clean[col].replace(9, np.nan)
        
    # Códigos específicos de NS/NC del INDEC
    df_clean['PP3E_TOT'] = df_clean['PP3E_TOT'].replace(999, np.nan)     # Horas
    df_clean['PP04C'] = df_clean['PP04C'].replace(99, np.nan)            # Tamaño empresa
    df_clean['PP04D_COD'] = df_clean['PP04D_COD'].replace(99999, np.nan) # Ocupación
    df_clean['ESTADO'] = df_clean['ESTADO'].replace(0, np.nan)           # Entrevista no realizada
    
    # Valores "0" que significan "No corresponde"
    df_clean.loc[df_clean['CAT_OCUP'] == 0, 'CAT_OCUP'] = np.nan
    df_clean.loc[df_clean['PP07H'] == 0, 'PP07H'] = np.nan
    
    # Ingresos: Cualquier valor <= 0 (incluyendo el -9 de NS/NC) pasa a NaN
    df_clean.loc[df_clean['P21'] <= 0, 'P21'] = np.nan
    df_clean.loc[df_clean['P47T'] <= 0, 'P47T'] = np.nan

    # ---------------------------------------------------------
    # Limpieza outliers:
    # ---------------------------------------------------------
    
    # Nos quedamos estrictamente con la Población en Edad de Trabajar (PET)
    df_clean = df_clean[df_clean['CH06'] >= 15]

    # Límite físico humano de horas trabajadas (tope conservador)
    df_clean.loc[df_clean['PP3E_TOT'] > 120, 'PP3E_TOT'] = np.nan
    
    # Trimming (recorte) del 1% superior de la distribución de ingresos
    if df_clean['P21'].notna().sum() > 0:
        q99_p21 = df_clean['P21'].quantile(0.99)
        df_clean.loc[df_clean['P21'] > q99_p21, 'P21'] = np.nan
        
    if df_clean['P47T'].notna().sum() > 0:
        q99_p47t = df_clean['P47T'].quantile(0.99)
        df_clean.loc[df_clean['P47T'] > q99_p47t, 'P47T'] = np.nan
        
    return df_clean

# Aplicamos la limpieza a las bases
T424_v = reemplazar_nsnc_por_nan(T424_v)
T425_v = reemplazar_nsnc_por_nan(T425_v)

In [11]:
# Repetimos resumen descriptivo de las bases para observar posibles outliers ya sin los NS/NC
resumen_2024 = T424_v.describe().T
resumen_2025 = T425_v.describe().T

# Mostramos el resultado del 2024
print("Resumen estadístico EPH - Cuarto Trimestre 2024:")
display(resumen_2024) 

# Mostramos el resultado del 2025
print("\nResumen estadístico EPH - Cuarto Trimestre 2025:")
display(resumen_2025)

Resumen estadístico EPH - Cuarto Trimestre 2024:


,count,mean,std,min,25%,50%,75%,max
ANO4,37889.0,2024.000000,0.000000,2024.0,2024.0,2024.0,2024.0,2024.0
TRIMESTRE,37889.0,4.000000,0.000000,4.0,4.0,4.0,4.0,4.0
NRO_HOGAR,37889.0,1.033730,0.698977,1.0,1.0,1.0,1.0,72.0
PONDERA,37889.0,609.624614,778.611442,23.0,174.0,307.0,648.0,7989.0
CH04,37889.0,1.527303,0.499261,1.0,1.0,2.0,2.0,2.0
CH06,37889.0,43.586555,18.999866,15.0,27.0,42.0,58.0,103.0
CH07,37888.0,3.171558,1.653192,1.0,2.0,3.0,5.0,5.0
CH08,37889.0,2.079020,1.902164,1.0,1.0,1.0,4.0,123.0
NIVEL_ED,37889.0,3.913484,1.437338,1.0,3.0,4.0,5.0,7.0
ESTADO,37839.0,1.850340,0.972010,1.0,1.0,1.0,3.0,3.0



Resumen estadístico EPH - Cuarto Trimestre 2025:


,count,mean,std,min,25%,50%,75%,max
ANO4,35784.0,2025.000000,0.000000,2025.0,2025.0,2025.0,2025.0,2025.0
TRIMESTRE,35784.0,4.000000,0.000000,4.0,4.0,4.0,4.0,4.0
NRO_HOGAR,35784.0,1.039571,0.924262,1.0,1.0,1.0,1.0,71.0
PONDERA,35784.0,652.755729,848.712008,24.0,183.0,328.0,689.0,7954.0
CH04,35784.0,1.530153,0.499097,1.0,1.0,2.0,2.0,2.0
CH06,35784.0,44.108959,19.242213,15.0,28.0,42.0,59.0,101.0
CH07,35781.0,3.184595,1.650362,1.0,2.0,3.0,5.0,5.0
CH08,35784.0,2.098536,1.790486,1.0,1.0,1.0,4.0,23.0
NIVEL_ED,35784.0,3.891152,1.443251,1.0,3.0,4.0,5.0,7.0
ESTADO,35725.0,1.861190,0.971700,1.0,1.0,1.0,3.0,3.0


Con el objetivo de garantizar la robustez del análisis estadístico, se procedió a la detección y depuración de outliers e inconsistencias en las variables de interés. En primer lugar, se aplicó un filtro de edad para trabajar exclusivamente con la Población en Edad de Trabajar (≥15 años). Posteriormente, para las variables de ingresos (P21 y P47T), se eliminaron valores iguales o menores a cero y se aplicó un recorte en el percentil 99 de la distribución para mitigar el sesgo introducido por valores extremos que no reflejan la realidad promedio de los trabajadores. Finalmente, respecto a la variable de horas trabajadas (PP3E_TOT), se descartaron registros superiores a las 120 horas semanales por resultar físicamente inviables, asegurando así la integridad de los estimadores salariales y la comparabilidad de los resultados entre los períodos analizados.

In [18]:
#Para 2024, como ya limipiamos la base, tomamos como "respondieron" a todos aquellos que tengan valores no nulos
respondieron_2024 = T424_v[T424_v['ESTADO'].notna()].copy()
norespondieron_2024 = T424_v[T424_v['ESTADO'].isna()].copy()

#Para 2025
respondieron_2025 = T425_v[T425_v['ESTADO'].notna()].copy()
norespondieron_2025 = T425_v[T425_v['ESTADO'].isna()].copy()

In [19]:
#De quienes respondieron para cada año, nos quedamos solamente con los ocupados y creamos la subbase OCUPADOS
ocupados_2024 = respondieron_2024[respondieron_2024['ESTADO'] == 1].copy()
ocupados_2025 = respondieron_2025[respondieron_2025['ESTADO'] == 1].copy()